In [ ]:
import numpy as np
import pandas as pd
#Validation Set by model (First, 3-zone sampling)
# -----------------------------
# 1) 유사도 행렬 CSV 불러오기
# -----------------------------
def load_similarity_matrix(path):
    df = pd.read_csv(path, index_col=0)
    print("Similarity matrix loaded:", path, df.shape)
    return df


# ----------------------------------
# 2) Flatten: (FAQ idx, Q idx, sim)
# ----------------------------------
def flatten_similarity_matrix(sim_matrix):
    pairs = []
    similarities = []

    for a_idx in range(sim_matrix.shape[0]):
        for b_idx in range(sim_matrix.shape[1]):
            sim = sim_matrix.iat[a_idx, b_idx]
            pairs.append((a_idx, b_idx))
            similarities.append(sim)

    return np.array(similarities), pairs


# -------------------------------------------------
# 3) 모델별 3구간 Validation Set 생성
# -------------------------------------------------
def build_three_zone_validation_csv(
    similarities,
    pairs,
    A_questions,
    B_questions,
    model_name,
    kneedle_threshold,
    q90_threshold,
    sample_n=30,
    out_path=None,
    random_seed=42
):
    df = pd.DataFrame(pairs, columns=["A_idx", "B_idx"])
    df["similarity"] = similarities

    df["A_text"] = df["A_idx"].apply(lambda i: A_questions[i])
    df["B_text"] = df["B_idx"].apply(lambda i: B_questions[i])

    # -----------------------------
    # 3개 구간 분리
    # -----------------------------
    similar = df[df["similarity"] >= q90_threshold].copy()
    grayzone = df[
        (df["similarity"] > kneedle_threshold) &
        (df["similarity"] < q90_threshold)
    ].copy()
    dissimilar = df[df["similarity"] <= kneedle_threshold].copy()

    print(f"\n[{model_name}]")
    print(f"Kneedle threshold: {kneedle_threshold}")
    print(f"90% threshold: {q90_threshold}")
    print(f"Similar candidates: {len(similar)}")
    print(f"Grayzone candidates: {len(grayzone)}")
    print(f"Dissimilar candidates: {len(dissimilar)}")

    # -----------------------------
    # 각 구간에서 무작위 추출
    # -----------------------------
    similar_sample = similar.sample(
        n=min(sample_n, len(similar)),
        random_state=random_seed
    )
    grayzone_sample = grayzone.sample(
        n=min(sample_n, len(grayzone)),
        random_state=random_seed
    )
    dissimilar_sample = dissimilar.sample(
        n=min(sample_n, len(dissimilar)),
        random_state=random_seed
    )

    similar_sample["group"] = "similar_above_q90"
    grayzone_sample["group"] = "grayzone_between_kneedle_q90"
    dissimilar_sample["group"] = "dissimilar_below_kneedle"

    val_df = pd.concat(
        [similar_sample, grayzone_sample, dissimilar_sample],
        axis=0
    ).reset_index(drop=True)

    val_df["model"] = model_name
    val_df["kneedle_threshold"] = kneedle_threshold
    val_df["q90_threshold"] = q90_threshold
    val_df["label"] = ""

    if out_path is None:
        out_path = f"validation_{model_name}_three_zone.csv"

    val_df.to_csv(out_path, index=False, encoding="utf-8-sig")
    print(f"Validation CSV saved: {out_path} ({len(val_df)} rows)")

    return val_df


# --------------------------------------
# 4) 전체 실행
# --------------------------------------
if __name__ == "__main__":

    # 질문 텍스트 (Path)
    A_questions = pd.read_csv(
        r"Projects\ADHD2\Data\Aset.csv"
    )["QuestionA"].tolist()

    B_questions = pd.read_csv(
        r"Projects\ADHD2\Data\Bset.csv"
    )["question_full"].tolist()

    # 모델별 설정 (Path)
    model_configs = {
        "mE5L": {
            "matrix_path":  r"Projects\ADHD2\analysis_results\mE5L\best\mE5L_best_similarity_matrix_clipped.csv",
            "kneedle_threshold": 0.8738,
            "q90_threshold": 0.8879
        },
        "MBERT": {
            "matrix_path": r"Projects\ADHD2\analysis_results\MBERT\best\MBERT_best_similarity_matrix_clipped.csv",
            "kneedle_threshold": 0.4742,
            "q90_threshold": 0.7522
        },
        "LaBSE": {
            "matrix_path": r"Projects\ADHD2\analysis_results\LaBSE\best\LaBSE_best_similarity_matrix_clipped.csv",
            "kneedle_threshold": 0.4567,
            "q90_threshold": 0.5912
        }
    }

    all_validation_dfs = []

    for model_name, config in model_configs.items():

        sim_matrix = load_similarity_matrix(config["matrix_path"])
        similarities, pairs = flatten_similarity_matrix(sim_matrix)

        val_df = build_three_zone_validation_csv(
            similarities=similarities,
            pairs=pairs,
            A_questions=A_questions,
            B_questions=B_questions,
            model_name=model_name,
            kneedle_threshold=config["kneedle_threshold"],
            q90_threshold=config["q90_threshold"],
            sample_n=30,
            out_path=rf"Projects\ADHD2\valid\validation_{model_name}_three_zone.csv",
            random_seed=42
        )

        all_validation_dfs.append(val_df)

    total_val_df = pd.concat(all_validation_dfs, axis=0).reset_index(drop=True)

    total_val_df.to_csv(
        r"Projects\ADHD2\valid\validation_all_models_three_zone.csv",
        index=False,
        encoding="utf-8-sig"
    )

    print("Done.")

In [ ]:
import pandas as pd
from sklearn.metrics import cohen_kappa_score
# Cohen's kappa score 
path = r"Projects\ADHD2\valid\all_model_validation_check.csv" #change path

df = pd.read_csv(path)

label_col_1 = "label 1"
label_col_2 = "label 2"

# 결측 제거
valid_df = df.dropna(subset=[label_col_1, label_col_2]).copy()

# 라벨을 정수형으로 통일
valid_df[label_col_1] = valid_df[label_col_1].astype(int)
valid_df[label_col_2] = valid_df[label_col_2].astype(int)

# 전체 Cohen's kappa
overall_kappa = cohen_kappa_score(
    valid_df[label_col_1],
    valid_df[label_col_2]
)

print(f"Overall Cohen's kappa: {overall_kappa:.4f}")

# 모델별 Cohen's kappa
model_kappa = (
    valid_df
    .groupby("model")
    .apply(lambda x: cohen_kappa_score(x[label_col_1], x[label_col_2]))
    .reset_index(name="cohen_kappa")
)

print(model_kappa)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    f1_score,
    recall_score,
    precision_score,
    accuracy_score,
)
# manual Validation 
validation_path = r"Projects\ADHD2\valid\all_model_validation_check.csv"  #path change

model_configs = {
    "mE5L": {
        "kneedle_threshold": 0.8738,
        "q90_threshold": 0.8879,
    },
    "MBERT": {
        "kneedle_threshold": 0.4742,
        "q90_threshold": 0.7522,
    },
    "LaBSE": {
        "kneedle_threshold": 0.4567,
        "q90_threshold": 0.5912,
    },
}

df = pd.read_csv(validation_path)

df = df.dropna(subset=["final_label", "similarity", "model"]).copy()
df["final_label"] = df["final_label"].astype(int)

def evaluate_threshold(data, threshold):
    y_true = data["final_label"]
    y_pred = (data["similarity"] >= threshold).astype(int)

    return {
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "accuracy": accuracy_score(y_true, y_pred),
    }

def grid_search_best_threshold(data):
    thresholds = np.sort(data["similarity"].unique())

    rows = []
    for threshold in thresholds:
        scores = evaluate_threshold(data, threshold)
        rows.append({
            "threshold": threshold,
            **scores,
        })

    grid_df = pd.DataFrame(rows)

    best_row = (
        grid_df
        .sort_values(["f1", "threshold"], ascending=[False, True])
        .iloc[0]
    )

    return float(best_row["threshold"]), grid_df

summary_rows = []
grid_results = []

for model_name, config in model_configs.items():
    model_df = df[df["model"] == model_name].copy()

    kneedle_threshold = config["kneedle_threshold"]
    q90_threshold = config["q90_threshold"]
    manual_threshold, grid_df = grid_search_best_threshold(model_df)

    for criterion, threshold in [
        ("kneedle", kneedle_threshold),
        ("q90", q90_threshold),
        ("manual_grid_search", manual_threshold),
    ]:
        scores = evaluate_threshold(model_df, threshold)

        summary_rows.append({
            "model": model_name,
            "criterion": criterion,
            "threshold": threshold,
            **scores,
            "n": len(model_df),
        })

    grid_df["model"] = model_name
    grid_results.append(grid_df)

summary_df = pd.DataFrame(summary_rows)

summary_df = summary_df[[
    "model", "criterion", "threshold",
    "f1", "recall", "precision", "accuracy"
]]

print(summary_df.round(4))

summary_df.to_csv(
    r"Projects\ADHD2\valid\threshold_evaluation_summary.csv",
    index=False,
    encoding="utf-8-sig"
)



In [ ]:
import math
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, f1_score
#Sample size estimation based on F1 score
calibration_path = r"C:\Users\ghldn\Projects\ADHD2\valid\all_model_validation_check.csv"

model_thresholds = {
    "mE5L": 0.8751,
    "MBERT": 0.5981,
    "LaBSE": 0.6471,
}

target_ci_width = 0.20
z = 1.96

df = pd.read_csv(calibration_path)

df = df.dropna(subset=["model", "similarity", "final_label"]).copy()
df["final_label"] = df["final_label"].astype(int)

df["threshold"] = df["model"].map(model_thresholds)
df["pred_label"] = (df["similarity"] >= df["threshold"]).astype(int)

y_true = df["final_label"]
y_pred = df["pred_label"]

tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

n = tp + fp + tn + fn
observed_f1 = f1_score(y_true, y_pred, zero_division=0)

p_tp = tp / n
p_fp = fp / n
p_tn = tn / n
p_fn = fn / n

denom = 2 * p_tp + p_fp + p_fn

# F1 = 2TP / (2TP + FP + FN)
# Delta method gradient
d_tp = 2 * (p_fp + p_fn) / (denom ** 2)
d_fp = -2 * p_tp / (denom ** 2)
d_tn = 0
d_fn = -2 * p_tp / (denom ** 2)

probs = np.array([p_tp, p_fp, p_tn, p_fn])
gradient = np.array([d_tp, d_fp, d_tn, d_fn])

# Multinomial delta-method variance constant
var_constant = np.sum(probs * gradient ** 2) - (np.sum(probs * gradient)) ** 2

required_n = math.ceil(
    ((2 * z * math.sqrt(var_constant)) / target_ci_width) ** 2
)

expected_se = math.sqrt(var_constant / required_n)
expected_ci_width = 2 * z * expected_se

print("=== F1-based sample size estimation ===")
print(f"Calibration N = {n}")
print(f"TP={tp}, FP={fp}, TN={tn}, FN={fn}")
print(f"Observed F1 = {observed_f1:.4f}")
print(f"Target 95% CI width <= {target_ci_width}")
print(f"Estimated minimum N = {required_n}")
print(f"Expected SE at N = {expected_se:.4f}")
print(f"Expected 95% CI width at N = {expected_ci_width:.4f}")

In [ ]:
import numpy as np
import pandas as pd
#Test Validation set 추출 코드
calibration_path = r"Projects\ADHD2\valid\all_model_validation_check.csv"
aset_path = r"Projects\ADHD2\Data\Aset.csv"
bset_path = r"Projects\ADHD2\Data\Bset.csv"

output_path = r"Projects\ADHD2\valid\test_validation_all_models_three_zone_excluding_calibration.csv"

model_configs = {
    "mE5L": {
        "matrix_path": r"Projects\ADHD2\analysis_results\mE5L\best\mE5L_best_similarity_matrix_clipped.csv",
        "kneedle_threshold": 0.8738,
        "q90_threshold": 0.8879,
    },
    "MBERT": {
        "matrix_path": r"Projects\ADHD2\analysis_results\MBERT\best\MBERT_best_similarity_matrix_clipped.csv",
        "kneedle_threshold": 0.4742,
        "q90_threshold": 0.7522,
    },
    "LaBSE": {
        "matrix_path": r"Projects\ADHD2\analysis_results\LaBSE\best\LaBSE_best_similarity_matrix_clipped.csv",
        "kneedle_threshold": 0.4567,
        "q90_threshold": 0.5912,
    },
}

samples_per_zone = 30
random_state = 42

zone_order = [
    "similar_above_q90",
    "grayzone_between_kneedle_q90",
    "dissimilar_below_kneedle",
]

# =========================
# 1. Deduplication
# =========================

cal_df = pd.read_csv(calibration_path)

cal_df["A_idx"] = cal_df["A_idx"].astype(int)
cal_df["B_idx"] = cal_df["B_idx"].astype(int)

exclude_pairs = set(
    zip(cal_df["A_idx"], cal_df["B_idx"])
)

print(f"Calibration pairs to exclude: {len(exclude_pairs)}")

# =========================
# 2. 질문 원문 매핑
# =========================

aset_df = pd.read_csv(aset_path)
bset_df = pd.read_csv(bset_path)

# Aset: Index 컬럼 기준
aset_df["A_idx"] = aset_df["Index"].astype(int)
a_text_map = aset_df.set_index("A_idx")["QuestionA"].to_dict()

# Bset: 별도 index 컬럼이 없으므로 row index를 B_idx로 사용
bset_df = bset_df.reset_index().rename(columns={"index": "B_idx"})
bset_df["B_idx"] = bset_df["B_idx"].astype(int)

# B 질문 원문은 question_full 우선, 없으면 title 사용
if "question_full" in bset_df.columns:
    b_text_source_col = "question_full"
elif "title" in bset_df.columns:
    b_text_source_col = "title"
else:
    raise ValueError("Bset.csv에서 question_full 또는 title 컬럼을 찾지 못했습니다.")

b_text_map = bset_df.set_index("B_idx")[b_text_source_col].to_dict()

print(f"A text map size: {len(a_text_map)}")
print(f"B text map size: {len(b_text_map)}")
print(f"B text source column: {b_text_source_col}")

# =========================
# 3. similarity matrix long 변환
# =========================

def matrix_to_long(matrix_path, model_name):
    matrix_df = pd.read_csv(matrix_path)

    a_col = matrix_df.columns[0]

    long_df = matrix_df.melt(
        id_vars=a_col,
        var_name="B_idx",
        value_name="similarity",
    )

    long_df = long_df.rename(columns={a_col: "A_idx"})

    long_df["A_idx"] = (
        long_df["A_idx"]
        .astype(str)
        .str.replace("A_", "", regex=False)
        .astype(int)
    )

    long_df["B_idx"] = (
        long_df["B_idx"]
        .astype(str)
        .str.replace("B_", "", regex=False)
        .astype(int)
    )

    long_df["model"] = model_name

    return long_df[["model", "A_idx", "B_idx", "similarity"]]


def assign_zone(df, kneedle_threshold, q90_threshold):
    df = df.copy()

    df["group"] = np.select(
        [
            df["similarity"] >= q90_threshold,
            (df["similarity"] >= kneedle_threshold) & (df["similarity"] < q90_threshold),
            df["similarity"] < kneedle_threshold,
        ],
        [
            "similar_above_q90",
            "grayzone_between_kneedle_q90",
            "dissimilar_below_kneedle",
        ],
        default="unassigned",
    )

    return df

# =========================
# 4. 모델별 zone당 30개 샘플링
# =========================

sampled_dfs = []
used_test_pairs = set()

for model_name, config in model_configs.items():
    model_df = matrix_to_long(
        matrix_path=config["matrix_path"],
        model_name=model_name,
    )

    # 기존 calibration 질문쌍 제외
    model_df["pair_key"] = list(zip(model_df["A_idx"], model_df["B_idx"]))
    model_df = model_df[~model_df["pair_key"].isin(exclude_pairs)].copy()

    model_df = assign_zone(
        model_df,
        kneedle_threshold=config["kneedle_threshold"],
        q90_threshold=config["q90_threshold"],
    )

    model_df["kneedle_threshold"] = config["kneedle_threshold"]
    model_df["q90_threshold"] = config["q90_threshold"]

    for group_name in zone_order:
        group_df = model_df[model_df["group"] == group_name].copy()

        # test set 내부에서도 같은 A_idx-B_idx 중복 방지
        group_df = group_df[~group_df["pair_key"].isin(used_test_pairs)].copy()

        available_n = len(group_df)

        if available_n < samples_per_zone:
            raise ValueError(
                f"{model_name} / {group_name}: "
                f"available {available_n}, required {samples_per_zone}"
            )

        sampled_group_df = group_df.sample(
            n=samples_per_zone,
            random_state=random_state,
        )

        used_test_pairs.update(sampled_group_df["pair_key"].tolist())
        sampled_dfs.append(sampled_group_df)

test_validation_df = pd.concat(sampled_dfs, ignore_index=True)

# =========================
# 5. 질문 원문 붙이고 저장
# =========================

test_validation_df["A_text"] = test_validation_df["A_idx"].map(a_text_map)
test_validation_df["B_text"] = test_validation_df["B_idx"].map(b_text_map)

missing_a_text = test_validation_df["A_text"].isna().sum()
missing_b_text = test_validation_df["B_text"].isna().sum()

print(f"Missing A_text: {missing_a_text}")
print(f"Missing B_text: {missing_b_text}")

test_validation_df["label 1"] = np.nan
test_validation_df["label 2"] = np.nan
test_validation_df["final_label"] = np.nan
test_validation_df["memo"] = ""

test_validation_df = test_validation_df.drop(columns=["pair_key"])

test_validation_df = test_validation_df[[
    "A_idx",
    "B_idx",
    "similarity",
    "A_text",
    "B_text",
    "group",
    "model",
    "kneedle_threshold",
    "q90_threshold",
    "label 1",
    "label 2",
    "final_label",
    "memo",
]]

# 검토 편의를 위해 섞기
test_validation_df = test_validation_df.sample(
    frac=1,
    random_state=random_state,
).reset_index(drop=True)

print("\n=== Sample count by model and group ===")
print(test_validation_df.groupby(["model", "group"]).size())

print(f"\nTotal sampled N = {len(test_validation_df)}")

test_validation_df.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig",
)

print(f"\nsaved: {output_path}")

In [ ]:
import pandas as pd
from sklearn.metrics import (
    f1_score,
    fbeta_score,
    recall_score,
    precision_score,
    accuracy_score,
)
#F2 추가값
validation_path = r"Projects\ADHD2\valid\all_model_validation_check.csv"

model_configs = {
    "mE5L": {
        "threshold": 0.8751,
    },
    "MBERT": {
        "threshold": 0.5981,
    },
    "LaBSE": {
        "threshold": 0.6471,
    },
}

df = pd.read_csv(validation_path)

df = df.dropna(subset=["model", "similarity", "final_label"]).copy()
df["final_label"] = df["final_label"].astype(int)

rows = []

for model_name, config in model_configs.items():
    model_df = df[df["model"] == model_name].copy()

    y_true = model_df["final_label"]
    y_pred = (model_df["similarity"] >= config["threshold"]).astype(int)

    rows.append({
        "model": model_name,
        "threshold": config["threshold"],
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "f2": fbeta_score(y_true, y_pred, beta=2, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "accuracy": accuracy_score(y_true, y_pred),
        "n": len(model_df),
    })

result_df = pd.DataFrame(rows)

print(result_df.round(4))